# FENE-v2 Chain Visualization

Runs a single FENE-v2 chain simulation and visualises the slow field $u(\xi, \tau)$.

**Physics:**
- Force:  $F(r) = H r / (1 - (r/R)^2)$,  $H = c^2$,  $R = \varepsilon^2 \cdot 30$
- Strain: $r_n = q_{n+1} - q_n$
- EOM: $\ddot{q}_n = F(r_n) - F(r_{n-1})$

**Scaling ansatz (same as chain-KdV):**
- $r_n(t_{\rm lab}) = \varepsilon^2\, u(\xi, \tau)$
- $\xi = \varepsilon(n - c\, t_{\rm lab})$,  $\tau = \varepsilon^3 t_{\rm lab}$

Parameters match `kdv.ipynb` (`accurate_test` preset): $\varepsilon = 0.03$, $c = 1$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rootutils
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

rootutils.setup_root(".", indicator=".project-root", pythonpath=True)
from data_generation.fene.generate import FENEv2IC, simulate_fene_v2  # noqa: E402

In [ ]:
# ── Parameters (kdv.ipynb accurate_test preset) ────────────────────────────────
C = 1.0  # linear wave speed (H = c^2)
LX = 2 * np.pi  # domain length
N_CHAIN = 209  # chain sites -> eps = Lx / N_chain ≈ 0.03
N_KDV = 256  # output grid points
DT_KDV = 1e-3  # slow-time step
T = 1.0  # total slow time
SAVE_EVERY = 5  # save every 20 slow steps
CHAIN_SUBSTEPS = 1  # auto-increased for CFL
NK = 3
MAX_K = 6

eps = LX / N_CHAIN
H = C**2
R = eps**2 * 30
print(f"eps = {eps:.5f}")
print(f"H = c^2 = {H:.4f},  R = eps^2*30 = {R:.6f}")

rng = np.random.default_rng(42)
u_ic = FENEv2IC(LX, Nk=NK, max_k=MAX_K)
u_ic.reset(rng)
print("IC:", u_ic)

In [ ]:
# ── Run simulation ─────────────────────────────────────────────────────────────
t_slow, xi_grid, u_series, r_max_seen = simulate_fene_v2(
    u_ic,
    c=C,
    eps=eps,
    Lx=LX,
    N_chain=N_CHAIN,
    N_kdv=N_KDV,
    dt_kdv=DT_KDV,
    T=T,
    chain_substeps=CHAIN_SUBSTEPS,
    save_every=SAVE_EVERY,
)

print(f"u_series shape : {u_series.shape}  (frames, N_kdv)")
print(f"t_slow range   : [{t_slow[0]:.4f}, {t_slow[-1]:.4f}]")
print(f"u range        : [{u_series.min():.4f}, {u_series.max():.4f}]")
print(f"max|r/R| seen  : {r_max_seen:.4f}  (must be < 1)")

In [ ]:
# ── FENE-v2 force law vs linearised (harmonic) ─────────────────────────────────
r_plot = np.linspace(-0.95 * R, 0.95 * R, 500)
F_fene = H * r_plot / (1 - (r_plot / R) ** 2)
F_lin = H * r_plot  # harmonic limit

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r_plot, F_fene, lw=2, label=r"FENE-v2: $Hr/(1-(r/R)^2)$")
ax.plot(r_plot, F_lin, "--", lw=1.5, label=r"harmonic: $Hr$")
ax.axvline(-R, color="gray", ls=":", lw=1, label=r"$\pm R$")
ax.axvline(R, color="gray", ls=":", lw=1)
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$F(r)$")
ax.set_title(rf"FENE-v2 force  ($H={H:.2f}$, $R={R:.4f}$)")
ax.legend()
ax.set_ylim(-5 * H * R, 5 * H * R)
fig.tight_layout()
plt.show()

In [ ]:
# ── Static snapshots: initial, middle, final ───────────────────────────────────
n_frames = u_series.shape[0]
snap_idx = [0, n_frames // 2, n_frames - 1]

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, idx in zip(axes, snap_idx):
    ax.plot(xi_grid, u_series[idx], lw=1.5)
    ax.set_title(rf"$\tau$ = {t_slow[idx]:.3f}")
    ax.set_xlabel(r"$\xi$")
    ax.set_xlim(xi_grid[0], xi_grid[-1])
axes[0].set_ylabel(r"$u(\xi, \tau)$")
fig.suptitle("FENE-v2: slow field $u(\\xi,\\tau)$ snapshots", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── Animation ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
(line,) = ax.plot(xi_grid, u_series[0], lw=1.5, color="C0")
title = ax.set_title(rf"$\tau$ = {t_slow[0]:.4f}")
ax.set_xlabel(r"$\xi$")
ax.set_ylabel(r"$u(\xi, \tau)$")
ax.set_xlim(xi_grid[0], xi_grid[-1])
ypad = 0.1 * float(np.abs(u_series).max())
ax.set_ylim(u_series.min() - ypad, u_series.max() + ypad)
fig.tight_layout()


def update(frame):
    line.set_ydata(u_series[frame])
    title.set_text(rf"$\tau$ = {t_slow[frame]:.4f}")
    return line, title


ani = FuncAnimation(fig, update, frames=n_frames, interval=80, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
# ── Space-time heatmap ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
vmax = float(np.max(np.abs(u_series)))
im = ax.imshow(
    u_series,
    aspect="auto",
    origin="lower",
    extent=[xi_grid[0], xi_grid[-1], t_slow[0], t_slow[-1]],
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
ax.set_xlabel(r"$\xi$")
ax.set_ylabel(r"$\tau$")
ax.set_title(r"FENE-v2: space-time plot of $u(\xi, \tau)$")
fig.colorbar(im, ax=ax, label=r"$u$")
fig.tight_layout()
plt.show()